# Build Layer 2 dataset

In [ ]:
from __future__ import annotations

import ast
import json
import logging
import math
import os
import random
import re
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 200)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BUILD_TS_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
print("Build timestamp (UTC):", BUILD_TS_UTC)
print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)

In [ ]:
def find_project_root(start: Optional[Path] = None) -> Path:
    candidates = []
    if start is None:
        start = Path.cwd()
    for base in [start, *start.parents, Path("/workspace"), Path("/mnt/data")]:
        if not base.exists():
            continue
        score = 0
        if (base / "configs" / "config.yaml").exists():
            score += 3
        if (base / "output").exists():
            score += 2
        if (base / "notebooks").exists():
            score += 1
        if score > 0:
            candidates.append((score, base))
    if candidates:
        candidates.sort(key=lambda x: (-x[0], len(str(x[1]))))
        return candidates[0][1]
    return start

PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / "output"
LOG_ROOT = OUTPUT_ROOT / "logs"
LAYER2_OUT = OUTPUT_ROOT / "09_layer2_dataset"

for p in [OUTPUT_ROOT, LOG_ROOT, LAYER2_OUT]:
    p.mkdir(parents=True, exist_ok=True)

REFINED_DIRS = {
    "manifest": OUTPUT_ROOT / "00_manifest",
    "segments": OUTPUT_ROOT / "02_segments_refined",
    "tokens": OUTPUT_ROOT / "05_tokens_refined",
    "dictionary": OUTPUT_ROOT / "06_dictionary_refined",
    "audits": OUTPUT_ROOT / "07_audits_refined",
}

PROJECT_ROOT, LAYER2_OUT

In [ ]:
LOG_FILE = LOG_ROOT / "08_build_layer2_dataset_fixed.log"

logger = logging.getLogger("layer2_dataset_fixed")
logger.setLevel(logging.INFO)
logger.handlers = []

fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

fh = logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8")
fh.setLevel(logging.INFO)
fh.setFormatter(fmt)
logger.addHandler(fh)

sh = logging.StreamHandler(sys.stdout)
sh.setLevel(logging.INFO)
sh.setFormatter(fmt)
logger.addHandler(sh)

logger.info("Started 08_build_layer2_dataset_fixed")
logger.info("PROJECT_ROOT=%s", PROJECT_ROOT)
logger.info("LAYER2_OUT=%s", LAYER2_OUT)

In [ ]:
def normalize_text(x: Any) -> Optional[str]:
    if x is None:
        return None
    if isinstance(x, float) and np.isnan(x):
        return None
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none", "null"}:
        return None
    s = s.replace("\\", "/")
    s = re.sub(r"/+", "/", s)
    return s.strip()

def canonical_token(x: Any) -> Optional[str]:
    s = normalize_text(x)
    if s is None:
        return None
    s = s.strip().lower()
    m = re.fullmatch(r"u?0*(\d+)", s)
    if m:
        return f"u{int(m.group(1)):02d}"
    return s

def stem_like(x: Any) -> Optional[str]:
    s = normalize_text(x)
    if s is None:
        return None
    p = Path(s)
    name = p.name if p.name else s
    if "." in name:
        name = Path(name).stem
    return name.lower()

def try_read_table(path: Path) -> pd.DataFrame:
    suf = path.suffix.lower()
    if suf == ".csv":
        return pd.read_csv(path)
    if suf == ".gz" and path.name.endswith(".csv.gz"):
        return pd.read_csv(path, compression="gzip")
    if suf == ".parquet":
        return pd.read_parquet(path)
    if suf in {".jsonl", ".ndjson"}:
        rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return pd.DataFrame(rows)
    if suf == ".json":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            return pd.DataFrame(data)
        if isinstance(data, dict):
            return pd.json_normalize(data)
    raise ValueError(f"Unsupported file type: {path}")

def safe_json_dump(obj: Any, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def save_df(df: pd.DataFrame, parquet_path: Path, csv_gz_path: Path, index: bool = False) -> Dict[str, Any]:
    parquet_ok = None
    try:
        df.to_parquet(parquet_path, index=index)
        parquet_ok = str(parquet_path)
    except Exception as e:
        logger.warning("Parquet save failed for %s: %s", parquet_path, e)
    df.to_csv(csv_gz_path, index=index, compression="gzip")
    return {"rows": int(len(df)), "parquet": parquet_ok, "csv_gz": str(csv_gz_path)}

def first_matching_column(df: pd.DataFrame, keyword_groups: Sequence[Sequence[str]]) -> Optional[str]:
    cols = list(df.columns)
    lowered = {c: c.lower() for c in cols}
    for keys in keyword_groups:
        for c, cl in lowered.items():
            if all(k.lower() in cl for k in keys):
                return c
    return None

def add_record_key(df: pd.DataFrame, label: str) -> pd.DataFrame:
    df = df.copy()
    candidates = [
        "clip_id", "clip_uid", "recording_id", "audio_path", "relative_path", "path", "file_name", "filename"
    ]
    source_col = None
    for c in candidates:
        if c in df.columns:
            source_col = c
            break
    if source_col is None:
        raise ValueError(f"Could not infer record key column for {label}")
    df["_record_key"] = df[source_col].map(stem_like)
    if df["_record_key"].isna().any():
        fallback = df[source_col].map(normalize_text)
        df.loc[df["_record_key"].isna(), "_record_key"] = fallback[df["_record_key"].isna()]
    return df

def parse_maybe_list(v: Any) -> List[Any]:
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return []
    if isinstance(v, list):
        return v
    if isinstance(v, tuple):
        return list(v)
    if isinstance(v, str):
        s = v.strip()
        if not s:
            return []
        if s.startswith("[") and s.endswith("]"):
            try:
                z = json.loads(s)
                if isinstance(z, list):
                    return z
            except Exception:
                pass
            try:
                z = ast.literal_eval(s)
                if isinstance(z, list):
                    return z
            except Exception:
                pass
        if "," in s:
            return [x.strip() for x in s.split(",") if x.strip()]
        return [x for x in s.split() if x]
    return [v]

def robust_entropy(values: Sequence[Any]) -> float:
    vals = [v for v in values if v is not None]
    if not vals:
        return 0.0
    counts = Counter(vals)
    tot = sum(counts.values())
    probs = np.array([v / tot for v in counts.values()], dtype=float)
    return float(-(probs * np.log2(np.clip(probs, 1e-12, 1.0))).sum())

In [ ]:
def choose_first_existing(paths: Sequence[Path]) -> Optional[Path]:
    for p in paths:
        if p.exists():
            return p
    return None

def find_one(root: Path, patterns: Sequence[str]) -> Optional[Path]:
    if not root.exists():
        return None
    matches = []
    for pat in patterns:
        matches.extend(root.rglob(pat))
    matches = [m for m in matches if m.is_file()]
    if not matches:
        return None
    return sorted(set(matches), key=lambda p: (len(str(p)), str(p)))[0]

def find_many(root: Path, patterns: Sequence[str]) -> List[Path]:
    if not root.exists():
        return []
    matches = []
    for pat in patterns:
        matches.extend(root.rglob(pat))
    return sorted(set([m for m in matches if m.is_file()]), key=lambda p: str(p))

manifest_path = choose_first_existing([
    REFINED_DIRS["manifest"] / "manifest.csv",
    REFINED_DIRS["manifest"] / "manifest.parquet",
]) or find_one(REFINED_DIRS["manifest"], ["manifest*.csv", "manifest*.parquet"])

segments_path = choose_first_existing([
    REFINED_DIRS["segments"] / "segments.csv",
    REFINED_DIRS["segments"] / "segments.parquet",
]) or find_one(REFINED_DIRS["segments"], ["segments*.csv", "segments*.parquet"])

token_sequences_path = choose_first_existing([
    REFINED_DIRS["tokens"] / "token_sequences.jsonl",
    REFINED_DIRS["tokens"] / "token_sequences.json",
]) or find_one(REFINED_DIRS["tokens"], ["*token*sequence*.jsonl", "*token*sequence*.json"])

token_counts_path = choose_first_existing([
    REFINED_DIRS["tokens"] / "token_counts_by_clip.csv",
    REFINED_DIRS["tokens"] / "token_counts_by_clip.parquet",
]) or find_one(REFINED_DIRS["tokens"], ["*token*count*.csv", "*token*count*.parquet"])

global_freq_path = choose_first_existing([
    REFINED_DIRS["tokens"] / "global_token_frequencies.csv",
    REFINED_DIRS["tokens"] / "global_token_frequencies.parquet",
]) or find_one(REFINED_DIRS["tokens"], ["*global*token*freq*.csv", "*global*token*freq*.parquet"])

dictionary_files = find_many(REFINED_DIRS["dictionary"], ["*.csv", "*.parquet"])
audit_files = find_many(REFINED_DIRS["audits"], ["*.csv", "*.parquet"])

for name, p in {
    "manifest": manifest_path,
    "segments": segments_path,
    "token_sequences": token_sequences_path,
    "token_counts": token_counts_path,
    "global_freq": global_freq_path,
}.items():
    logger.info("%s -> %s", name, p)
logger.info("dictionary_files=%s", [str(x.name) for x in dictionary_files])
logger.info("audit_files=%s", [str(x.name) for x in audit_files])

In [ ]:
if manifest_path is None:
    raise FileNotFoundError("Manifest file not found in output/00_manifest")

manifest = add_record_key(try_read_table(manifest_path), "manifest")
segments = add_record_key(try_read_table(segments_path), "segments") if segments_path else pd.DataFrame()
token_counts_raw = add_record_key(try_read_table(token_counts_path), "token_counts_by_clip") if token_counts_path else pd.DataFrame()
global_freq_raw = try_read_table(global_freq_path) if global_freq_path else pd.DataFrame()

logger.info("manifest shape=%s", manifest.shape)
logger.info("segments shape=%s", segments.shape)
logger.info("token_counts_raw shape=%s", token_counts_raw.shape)
logger.info("global_freq_raw shape=%s", global_freq_raw.shape)

manifest.head(2)

In [ ]:
def extract_sequence_container(record: Dict[str, Any]) -> Any:
    for key in ["token_sequence", "tokens", "sequence", "unit_sequence", "units", "token_events", "events", "segments"]:
        if key in record and record[key] is not None:
            return record[key]
    return None

def parse_token_records(path: Optional[Path]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if path is None:
        logger.warning("Token sequence file missing")
        return pd.DataFrame(), pd.DataFrame()

    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        records = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
    else:
        with open(path, "r", encoding="utf-8") as f:
            loaded = json.load(f)
        records = loaded if isinstance(loaded, list) else [loaded]

    event_rows = []
    clip_rows = []

    for rec in records:
        key_src = (
            rec.get("clip_id")
            or rec.get("clip_uid")
            or rec.get("recording_id")
            or rec.get("audio_path")
            or rec.get("relative_path")
            or rec.get("path")
            or rec.get("file_name")
            or rec.get("filename")
        )
        record_key = stem_like(key_src)
        if record_key is None:
            continue

        container = extract_sequence_container(rec)
        seq_tokens = []
        seq_indices = []

        if isinstance(container, list):
            for idx, item in enumerate(container):
                if isinstance(item, dict):
                    tok = (
                        item.get("token")
                        or item.get("unit_id")
                        or item.get("unit")
                        or item.get("label")
                        or item.get("cluster_id")
                        or item.get("unit_index")
                    )
                    pos = item.get("position", idx)
                    start = item.get("start_sec") or item.get("start")
                    end = item.get("end_sec") or item.get("end")
                    dur = item.get("duration_sec") or item.get("duration")
                else:
                    tok = item
                    pos = idx
                    start = end = dur = None

                tok = canonical_token(tok)
                if tok is None:
                    continue
                seq_tokens.append(tok)
                seq_indices.append(idx)

                event_rows.append({
                    "_record_key": record_key,
                    "position": pos,
                    "token": tok,
                    "start_sec": pd.to_numeric(start, errors="coerce"),
                    "end_sec": pd.to_numeric(end, errors="coerce"),
                    "duration_sec": pd.to_numeric(dur, errors="coerce"),
                })
        else:
            simple_seq = parse_maybe_list(
                rec.get("sequence_unit_ids")
                or rec.get("sequence_unit_indices")
                or rec.get("token_sequence")
                or rec.get("tokens")
                or rec.get("units")
            )
            for idx, tok in enumerate(simple_seq):
                tok = canonical_token(tok)
                if tok is None:
                    continue
                seq_tokens.append(tok)
                seq_indices.append(idx)
                event_rows.append({
                    "_record_key": record_key,
                    "position": idx,
                    "token": tok,
                    "start_sec": np.nan,
                    "end_sec": np.nan,
                    "duration_sec": np.nan,
                })

        clip_rows.append({
            "_record_key": record_key,
            "sequence_unit_ids": seq_tokens,
            "sequence_unit_indices": list(range(len(seq_tokens))),
            "_raw_sequence_len": len(seq_tokens),
            "clip_duration_sec": pd.to_numeric(rec.get("clip_duration_sec") or rec.get("duration_sec"), errors="coerce"),
            "metadata": json.dumps(rec.get("metadata", {}), ensure_ascii=False) if isinstance(rec.get("metadata"), dict) else rec.get("metadata"),
        })

    token_events = pd.DataFrame(event_rows)
    clip_meta = pd.DataFrame(clip_rows).drop_duplicates(subset=["_record_key"])
    return token_events, clip_meta

token_events, token_sequence_meta = parse_token_records(token_sequences_path)
logger.info("token_events shape=%s", token_events.shape)
logger.info("token_sequence_meta shape=%s", token_sequence_meta.shape)
token_events.head(3)

In [ ]:
if (token_events is None or token_events.empty) and not segments.empty:
    logger.info("Token events empty; falling back to segments table.")
    token_col = first_matching_column(segments, [["token"], ["unit"], ["cluster", "id"], ["label"]])
    pos_col = first_matching_column(segments, [["position"], ["segment", "index"], ["order"]])
    start_col = first_matching_column(segments, [["start"]])
    end_col = first_matching_column(segments, [["end"]])
    dur_col = first_matching_column(segments, [["duration"]])

    if token_col is None:
        raise ValueError("Could not recover token events from segments")

    token_events = pd.DataFrame({
        "_record_key": segments["_record_key"],
        "position": pd.to_numeric(segments[pos_col], errors="coerce") if pos_col else segments.groupby("_record_key").cumcount(),
        "token": segments[token_col].map(canonical_token),
        "start_sec": pd.to_numeric(segments[start_col], errors="coerce") if start_col else np.nan,
        "end_sec": pd.to_numeric(segments[end_col], errors="coerce") if end_col else np.nan,
        "duration_sec": pd.to_numeric(segments[dur_col], errors="coerce") if dur_col else np.nan,
    }).dropna(subset=["token"]).reset_index(drop=True)

token_events = token_events.dropna(subset=["_record_key", "token"]).copy()
token_events["position"] = pd.to_numeric(token_events["position"], errors="coerce")
token_events["duration_sec"] = pd.to_numeric(token_events["duration_sec"], errors="coerce")
logger.info("final token_events shape=%s", token_events.shape)

In [ ]:
def build_token_count_features(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    if df is None or df.empty:
        return pd.DataFrame(columns=["_record_key"]), {"layout": "empty"}

    df = df.copy()
    protected = [c for c in df.columns if c.startswith("_")]
    for c in ["clip_id", "clip_uid", "recording_id", "audio_path", "relative_path", "path", "file_name", "filename"]:
        if c in df.columns:
            protected.append(c)

    token_col = first_matching_column(df, [["token"], ["unit_id"], ["unit"], ["cluster", "id"], ["label"], ["unit_index"]])
    count_col = first_matching_column(df, [["count"], ["frequency"], ["n_", "segments"]])

    if token_col is not None and count_col is not None and df["_record_key"].duplicated().any():
        tmp = df[["_record_key", token_col, count_col]].copy()
        tmp["token"] = tmp[token_col].map(canonical_token)
        tmp["count"] = pd.to_numeric(tmp[count_col], errors="coerce").fillna(0.0)
        tmp = tmp.dropna(subset=["_record_key", "token"])
        tmp = tmp.groupby(["_record_key", "token"], as_index=False)["count"].sum()
        wide = tmp.pivot(index="_record_key", columns="token", values="count").fillna(0.0)
        wide.columns = [f"tokcount__{c}" for c in wide.columns]
        wide = wide.reset_index()
        meta = {
            "layout": "long_to_wide",
            "token_col": token_col,
            "count_col": count_col,
            "n_features": int(len(wide.columns) - 1),
        }
        return wide, meta

    candidate_numeric = [c for c in df.columns if c not in protected and pd.api.types.is_numeric_dtype(df[c])]
    token_like_cols = [c for c in candidate_numeric if canonical_token(c) is not None and c.lower() not in {"count"}]
    if token_like_cols:
        wide = df[["_record_key"] + token_like_cols].copy()
        rename_map = {c: f"tokcount__{canonical_token(c)}" for c in token_like_cols}
        wide = wide.rename(columns=rename_map)
        wide = wide.groupby("_record_key", as_index=False).sum(numeric_only=True)
        meta = {
            "layout": "wide_token_columns",
            "n_features": int(len(wide.columns) - 1),
            "columns_preview": token_like_cols[:20],
        }
        return wide, meta

    return pd.DataFrame({"_record_key": sorted(df["_record_key"].dropna().astype(str).unique())}), {
        "layout": "no_token_count_features"
    }

token_count_features, token_count_meta = build_token_count_features(token_counts_raw)
logger.info("token_count_features shape=%s meta=%s", token_count_features.shape, token_count_meta)
token_count_features.head(3)

In [ ]:
def build_global_freq_table(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame(columns=["token", "global_token_frequency"])
    token_col = first_matching_column(df, [["token"], ["unit_id"], ["unit"], ["cluster", "id"], ["label"], ["unit_index"]])
    freq_col = first_matching_column(df, [["global", "frequency"], ["global", "count"], ["count"], ["frequency"], ["proportion"]])
    if token_col is None or freq_col is None:
        return pd.DataFrame(columns=["token", "global_token_frequency"])
    out = df[[token_col, freq_col]].copy()
    out["token"] = out[token_col].map(canonical_token)
    out["global_token_frequency"] = pd.to_numeric(out[freq_col], errors="coerce")
    out = out.dropna(subset=["token"]).groupby("token", as_index=False)["global_token_frequency"].mean()
    return out

global_freq = build_global_freq_table(global_freq_raw)
logger.info("global_freq shape=%s", global_freq.shape)
global_freq.head(3)

In [ ]:
def load_dictionary_tables(files: Sequence[Path]) -> Dict[str, pd.DataFrame]:
    out = {}
    for fp in files:
        try:
            out[fp.stem] = try_read_table(fp)
            logger.info("Loaded dictionary table %s shape=%s", fp.stem, out[fp.stem].shape)
        except Exception as e:
            logger.warning("Failed dictionary file %s: %s", fp, e)
    return out

dictionary_tables = load_dictionary_tables(dictionary_files)

unit_inventory = dictionary_tables.get("unit_inventory")
if unit_inventory is None:
    for name, df in dictionary_tables.items():
        if "inventory" in name.lower():
            unit_inventory = df
            break

def infer_tokenish_column(df: pd.DataFrame) -> Optional[str]:
    return first_matching_column(df, [["unit_id"], ["token"], ["unit"], ["cluster", "id"], ["label"], ["unit_index"]])

token_lexicon = pd.DataFrame(columns=["token"])
if unit_inventory is not None and not unit_inventory.empty:
    tok_col = infer_tokenish_column(unit_inventory)
    if tok_col is not None:
        token_lexicon = unit_inventory.copy()
        token_lexicon["token"] = token_lexicon[tok_col].map(canonical_token)
        token_lexicon = token_lexicon.dropna(subset=["token"]).drop_duplicates(subset=["token"]).reset_index(drop=True)

if token_lexicon.empty and not token_events.empty:
    token_lexicon = pd.DataFrame({"token": sorted(token_events["token"].dropna().astype(str).unique())})

logger.info("token_lexicon shape=%s", token_lexicon.shape)
token_lexicon.head(3)

In [ ]:
def infer_audit_field_from_name(name: str) -> str:
    n = name.lower()
    if "farm" in n:
        return "farm"
    if "zone" in n or "barn_zone" in n or "barnzone" in n:
        return "barn_zone"
    if "placement" in n:
        return "mic_placement_context"
    if "microphone" in n or re.search(r"\bmic\b", n):
        return "microphone"
    return "unknown"

def standardize_audit_table(fp: Path) -> Optional[pd.DataFrame]:
    try:
        df = try_read_table(fp)
    except Exception as e:
        logger.warning("Audit load failed for %s: %s", fp, e)
        return None
    if df.empty:
        return None

    tok_col = infer_tokenish_column(df)
    if tok_col is None:
        return None

    out = pd.DataFrame()
    out["token"] = df[tok_col].map(canonical_token)
    out["audit_field"] = infer_audit_field_from_name(fp.stem)

    level_col = first_matching_column(df, [["bias", "level"], ["flag"], ["severity"], ["level"]])
    dom_prop_col = first_matching_column(df, [["dominant", "proportion"], ["dominant", "purity"], ["purity"], ["proportion"]])

    if level_col is not None:
        out["bias_level"] = df[level_col].astype(str).str.lower()
        out["any_bias_flag"] = ~out["bias_level"].isin(["none", "nan", "null", ""])
        out["any_high_bias_flag"] = out["bias_level"].str.contains("high", na=False)
    else:
        out["any_bias_flag"] = True
        out["any_high_bias_flag"] = False

    if dom_prop_col is not None:
        out["dominant_context_proportion"] = pd.to_numeric(df[dom_prop_col], errors="coerce")

    out = out.dropna(subset=["token"]).drop_duplicates()
    return out

audit_frames = []
for fp in audit_files:
    std = standardize_audit_table(fp)
    if std is not None and not std.empty:
        audit_frames.append(std)

audit_long = pd.concat(audit_frames, ignore_index=True) if audit_frames else pd.DataFrame(
    columns=["token", "audit_field", "any_bias_flag", "any_high_bias_flag", "dominant_context_proportion"]
)

if not audit_long.empty:
    audit_summary = (
        audit_long.groupby("token", as_index=False)
        .agg(
            any_bias_flag=("any_bias_flag", "max"),
            any_high_bias_flag=("any_high_bias_flag", "max"),
            n_audit_rows=("token", "size"),
            n_flagged_rows=("any_bias_flag", "sum"),
            n_high_rows=("any_high_bias_flag", "sum"),
            audit_fields=("audit_field", lambda s: "|".join(sorted(set([str(x) for x in s if pd.notna(x)])))),
        )
    )
else:
    audit_summary = pd.DataFrame(columns=[
        "token", "any_bias_flag", "any_high_bias_flag", "n_audit_rows", "n_flagged_rows", "n_high_rows", "audit_fields"
    ])

logger.info("audit_summary shape=%s", audit_summary.shape)
audit_summary.head(3)

In [ ]:
token_events = token_events.copy()

if not global_freq.empty:
    token_events = token_events.merge(global_freq, on="token", how="left")
if not audit_summary.empty:
    token_events = token_events.merge(audit_summary, on="token", how="left")

for c in ["any_bias_flag", "any_high_bias_flag"]:
    if c in token_events.columns:
        token_events[c] = token_events[c].fillna(False).astype(bool)

logger.info("enriched token_events shape=%s", token_events.shape)
token_events.head(3)

In [ ]:
def summarize_tokens(group: pd.DataFrame) -> pd.Series:
    seq = [x for x in group["token"].tolist() if pd.notna(x)]
    durations = pd.to_numeric(group["duration_sec"], errors="coerce").dropna()

    counts = Counter(seq)
    n_tokens = len(seq)
    n_unique = len(counts)
    top_token = counts.most_common(1)[0][0] if counts else None
    top_token_count = counts.most_common(1)[0][1] if counts else 0
    transitions = sum(1 for i in range(1, len(seq)) if seq[i] != seq[i - 1]) if len(seq) >= 2 else 0

    gf = pd.to_numeric(group.get("global_token_frequency", pd.Series(dtype=float)), errors="coerce").dropna()

    audit_fields_seen = sorted(set("|".join(group.get("audit_fields", pd.Series(dtype=str)).dropna().astype(str)).split("|")))
    audit_fields_seen = [x for x in audit_fields_seen if x]

    return pd.Series({
        "n_segments": int(n_tokens),
        "n_unique_tokens": int(n_unique),
        "sequence_token_string": " ".join(seq),
        "sequence_unit_ids": json.dumps(seq, ensure_ascii=False),
        "sequence_unit_indices": json.dumps(list(range(n_tokens)), ensure_ascii=False),
        "top_token": top_token,
        "top_token_count": int(top_token_count),
        "token_entropy": robust_entropy(seq),
        "transition_count": int(transitions),
        "transition_rate": float(transitions / max(n_tokens - 1, 1)) if n_tokens else 0.0,
        "duration_sum_sec": float(durations.sum()) if len(durations) else np.nan,
        "duration_mean_sec": float(durations.mean()) if len(durations) else np.nan,
        "duration_median_sec": float(durations.median()) if len(durations) else np.nan,
        "global_freq_mean": float(gf.mean()) if len(gf) else np.nan,
        "global_freq_median": float(gf.median()) if len(gf) else np.nan,
        "global_freq_min": float(gf.min()) if len(gf) else np.nan,
        "global_freq_max": float(gf.max()) if len(gf) else np.nan,
        "has_any_bias_flag_token": bool(group.get("any_bias_flag", pd.Series(dtype=bool)).fillna(False).astype(bool).any()),
        "has_any_high_bias_flag_token": bool(group.get("any_high_bias_flag", pd.Series(dtype=bool)).fillna(False).astype(bool).any()),
        "count_flagged_unique_tokens": int(group.loc[group.get("any_bias_flag", pd.Series(False, index=group.index)).fillna(False), "token"].nunique()),
        "count_high_flagged_unique_tokens": int(group.loc[group.get("any_high_bias_flag", pd.Series(False, index=group.index)).fillna(False), "token"].nunique()),
        "audit_fields_seen_in_clip": "|".join(audit_fields_seen),
    })

clip_sequence_summary = token_events.groupby("_record_key", as_index=False).apply(summarize_tokens, include_groups=False)
clip_sequence_summary = clip_sequence_summary.reset_index(drop=True)

logger.info("clip_sequence_summary shape=%s", clip_sequence_summary.shape)
clip_sequence_summary.head(3)

In [ ]:
manifest = manifest.copy()

manifest_keep_priority = [
    "_record_key",
    "clip_id", "clip_uid", "recording_id",
    "file_name", "filename", "relative_path", "audio_path", "path",
    "date", "day", "time", "datetime",
    "farm", "barn_zone", "barnzone", "barn", "zone",
    "microphone", "mic", "mic_id",
    "mic_placement_context", "placement_context", "mic_placement", "placement",
    "main_category", "maincategory", "category",
    "subcategory", "sub_category",
    "duration_sec", "clip_duration_sec",
]
manifest_keep = [c for c in manifest_keep_priority if c in manifest.columns]

for c in manifest.columns:
    cl = c.lower()
    if c not in manifest_keep and any(k in cl for k in ["date", "day", "time", "farm", "barn", "zone", "mic", "placement", "category", "context"]):
        manifest_keep.append(c)

manifest_core = manifest[manifest_keep].copy().drop_duplicates(subset=["_record_key"])

rename_aliases = {}
for c in manifest_core.columns:
    cl = c.lower()
    if c in {"barnzone", "barn", "zone"}:
        rename_aliases[c] = "barn_zone"
    elif c in {"mic", "mic_id"}:
        rename_aliases[c] = "microphone"
    elif c in {"placement_context", "mic_placement", "placement"}:
        rename_aliases[c] = "mic_placement_context"
    elif c in {"maincategory", "category"}:
        rename_aliases[c] = "main_category_final"
    elif c in {"main_category"}:
        rename_aliases[c] = "main_category_final"
    elif c in {"subcategory", "sub_category"}:
        rename_aliases[c] = "sub_category_final"
    elif c == "duration_sec":
        rename_aliases[c] = "clip_duration_sec"

manifest_core = manifest_core.rename(columns=rename_aliases)
manifest_core = manifest_core.loc[:, ~manifest_core.columns.duplicated()]

logger.info("manifest_core shape=%s", manifest_core.shape)
manifest_core.head(3)

In [ ]:
master = manifest_core.copy()
join_audit = []

def left_merge_with_audit(base: pd.DataFrame, other: pd.DataFrame, name: str) -> pd.DataFrame:
    if other is None or other.empty:
        join_audit.append({"table": name, "rows": 0, "matched_rows": 0, "coverage": 0.0})
        return base
    other = other.copy()
    if other["_record_key"].duplicated().any():
        raise ValueError(f"{name} is not clip-level; duplicate _record_key values detected")
    other_cols = [c for c in other.columns if c != "_record_key"]
    merged = base.merge(other, on="_record_key", how="left", suffixes=("", f"__{name}"))
    matched_rows = int(merged[other_cols].notna().any(axis=1).sum()) if other_cols else 0
    coverage = float(matched_rows / len(base)) if len(base) else 0.0
    join_audit.append({"table": name, "rows": int(len(other)), "matched_rows": matched_rows, "coverage": coverage})
    logger.info("Join %-22s matched %s/%s (%.3f)", name, matched_rows, len(base), coverage)
    return merged

if not token_sequence_meta.empty:
    token_sequence_meta = token_sequence_meta.drop_duplicates(subset=["_record_key"])

master = left_merge_with_audit(master, clip_sequence_summary, "clip_sequence_summary")
master = left_merge_with_audit(master, token_sequence_meta, "token_sequence_meta")
master = left_merge_with_audit(master, token_count_features, "token_count_features")

if "clip_duration_sec" in master.columns and "clip_duration_sec__token_sequence_meta" in master.columns:
    pass

master = master.loc[:, ~master.columns.duplicated()].copy()

logger.info("master shape=%s", master.shape)
master.head(3)

In [ ]:
expected_clips = int(manifest["_record_key"].nunique())
actual_clips = int(master["_record_key"].nunique())
print("Expected unique clips from manifest:", expected_clips)
print("Actual unique clips in master:", actual_clips)

if len(master) != expected_clips or actual_clips != expected_clips:
    raise AssertionError(
        f"Master table is not clip-level. Expected {expected_clips} rows/keys, got {len(master)} rows and {actual_clips} unique keys."
    )

print("Clip-level sanity check passed.")

In [ ]:
join_audit_df = pd.DataFrame(join_audit)

token_lexicon_out = token_lexicon.copy()
if not token_lexicon_out.empty and not audit_summary.empty:
    token_lexicon_out = token_lexicon_out.merge(audit_summary, on="token", how="left", suffixes=("", "__audit"))
token_lexicon_out = token_lexicon_out.drop_duplicates(subset=["token"]).sort_values("token").reset_index(drop=True)

master_report = save_df(
    master,
    parquet_path=LAYER2_OUT / "layer2_master_table.parquet",
    csv_gz_path=LAYER2_OUT / "layer2_master_table.csv.gz",
    index=False,
)
token_events_report = save_df(
    token_events,
    parquet_path=LAYER2_OUT / "layer2_token_events.parquet",
    csv_gz_path=LAYER2_OUT / "layer2_token_events.csv.gz",
    index=False,
)

token_lexicon_out.to_csv(LAYER2_OUT / "layer2_token_lexicon.csv", index=False)
join_audit_df.to_csv(LAYER2_OUT / "layer2_join_audit.csv", index=False)

build_report = {
    "build_ts_utc": BUILD_TS_UTC,
    "project_root": str(PROJECT_ROOT),
    "layer2_output_dir": str(LAYER2_OUT),
    "inputs": {
        "manifest_path": str(manifest_path) if manifest_path else None,
        "segments_path": str(segments_path) if segments_path else None,
        "token_sequences_path": str(token_sequences_path) if token_sequences_path else None,
        "token_counts_path": str(token_counts_path) if token_counts_path else None,
        "global_freq_path": str(global_freq_path) if global_freq_path else None,
        "dictionary_files": [str(x) for x in dictionary_files],
        "audit_files": [str(x) for x in audit_files],
    },
    "master_report": master_report,
    "token_events_report": token_events_report,
    "token_lexicon_rows": int(len(token_lexicon_out)),
    "join_audit": join_audit_df.to_dict(orient="records"),
    "token_count_meta": token_count_meta,
}
safe_json_dump(build_report, LAYER2_OUT / "layer2_build_report.json")
build_report

In [ ]:
diagnostics = {}

diagnostics["n_clips"] = int(len(master))
diagnostics["n_manifest_rows"] = int(len(manifest))
diagnostics["n_token_event_rows"] = int(len(token_events))
diagnostics["n_lexicon_rows"] = int(len(token_lexicon_out))

missing_fields = (
    master.isna().sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
missing_fields["missing_fraction"] = missing_fields["missing_count"] / max(len(master), 1)
diagnostics["missing_fields_top20"] = missing_fields.head(20).to_dict(orient="records")

label_distributions = {}
for label_col in ["main_category_final", "sub_category_final"]:
    if label_col in master.columns:
        label_distributions[label_col] = (
            master[label_col].fillna("MISSING").astype(str).value_counts(dropna=False).to_dict()
        )
diagnostics["label_distributions"] = label_distributions

per_farm_counts = {}
if "farm" in master.columns:
    per_farm_counts = master["farm"].fillna("MISSING").astype(str).value_counts(dropna=False).to_dict()
diagnostics["per_farm_counts"] = per_farm_counts

if "n_segments" in master.columns:
    seq_len = pd.to_numeric(master["n_segments"], errors="coerce").dropna()
else:
    seq_len = pd.Series(dtype=float)

diagnostics["token_sequence_length_stats"] = {
    "min": float(seq_len.min()) if len(seq_len) else None,
    "max": float(seq_len.max()) if len(seq_len) else None,
    "mean": float(seq_len.mean()) if len(seq_len) else None,
    "median": float(seq_len.median()) if len(seq_len) else None,
    "q25": float(seq_len.quantile(0.25)) if len(seq_len) else None,
    "q75": float(seq_len.quantile(0.75)) if len(seq_len) else None,
}

bias_cov = {}
for col in ["has_any_bias_flag_token", "has_any_high_bias_flag_token"]:
    if col in master.columns:
        s = master[col].fillna(False).astype(bool)
        bias_cov[col] = {
            "count_true": int(s.sum()),
            "fraction_true": float(s.mean()),
        }
diagnostics["bias_flag_coverage"] = bias_cov

safe_json_dump(diagnostics, LAYER2_OUT / "layer2_diagnostics.json")

print(json.dumps(diagnostics, indent=2)[:4000])